# Huấn luyện cổng MOESCORE trên Kaggle — XACLE 2027
Notebook chạy từ trên xuống: dữ liệu thật → bốn expert đã fine-tune, đóng băng → feature store → huấn luyện gate → chọn checkpoint theo SRCC validation → đánh giá validation/test. Tham số và loss dùng cấu hình đã chốt của repository hiện tại. Số liệu chỉ xuất hiện sau khi chạy thật; notebook không chứa kết quả mẫu.

Tham khảo phương pháp: [MOESCORE gốc](https://github.com/S-Orion/MOESCORE), bài báo `MOESCORE.pdf` trong repository. Xem hướng dẫn đầy đủ tại `docs/KAGGLE_GATING_GUIDE_VI.md`.


## 2. Cấu hình người dùng
Giữ `RUN_MODE='full'` cho quy trình hoàn chỉnh. Với mạng tắt, đặt `REPOSITORY_SOURCE` tới bản repository đã đính kèm và cung cấp cache Hugging Face. Đường dẫn checkpoint bên dưới là cố định.


In [ ]:
from pathlib import Path
RUN_MODE = "full"  # preflight, smoke, extract, train, resume, evaluate, full
PROJECT_ROOT = Path("/kaggle/working/MOESCORE")
FEATURE_STORE_ROOT = Path("/kaggle/working/moe_feature_store")
OUTPUT_DIR = Path("/kaggle/working/moe_gating")
INPUT_ROOT = Path("/kaggle/input")
DATASET_ROOT = None  # Ví dụ: "/kaggle/input/<slug>/XACLE_dataset"
WEIGHTS_ROOT = Path("/kaggle/input/datasets/qnv8071/xacle-weights/weights")
REPOSITORY_SOURCE = None  # Bản source đầy đủ tại Kaggle Input, cần cho chạy offline
REPO_URL = "https://github.com/vietquang8071/MOESCORE.git"
REPO_REVISION = None  # None: main hiện tại; hoặc SHA commit đã chứa các script notebook
HF_CACHE_DIR = None  # Hub cache: roberta-base, bert-base-uncased (E2), google-bert/bert-base-uncased (E3)
ALLOW_REPOSITORY_DOWNLOAD = True  # Đặt False khi chạy offline
ALLOW_CACHE_DOWNLOAD = False  # Cho phép chuẩn bị đúng các model ID mà từng expert gọi
EXISTING_FEATURE_STORE_ROOT = None  # Store hoàn chỉnh đính kèm từ phiên trước
RESTORE_OUTPUT_ROOT = None  # Thư mục chứa latest_gate.pt/best_gate.pt từ phiên trước
RESUME_CHECKPOINT = None  # None: OUTPUT_DIR/latest_gate.pt
INSTALL_REQUIREMENTS = True  # Cài requirements-kaggle.txt; False nếu môi trường đã được chuẩn bị
EXTRA_PACKAGES = []  # Gói bổ sung ngoài requirements; không thay torch/torchaudio/CUDA
WHEELHOUSE = None  # Thư mục wheel offline; không bắt buộc khi phụ thuộc đã có
EXTRACTION_BATCH_SIZES = {"expert1": 1, "expert2": 1, "expert3": 1, "expert4": 1}
SHARD_SIZE = 128
SEED = 2027
SAVE_FEATURE_STORE_ARCHIVE = False
assert RUN_MODE in {"preflight", "smoke", "extract", "train", "resume", "evaluate", "full"}
EXPERT_CHECKPOINTS = {
    "expert1": {"pretrained_checkpoint": WEIGHTS_ROOT / "pretrained/laion-clap.pt", "finetuned_checkpoint": WEIGHTS_ROOT / "finetuned_weights/expert1_best.pt"},
    "expert2": {"pretrained_checkpoint": WEIGHTS_ROOT / "pretrained/mga-clap.pt", "finetuned_checkpoint": WEIGHTS_ROOT / "finetuned_weights/expert2_best.pt"},
    "expert3": {"base_checkpoint": WEIGHTS_ROOT / "pretrained/checkpoint-30.pth", "finetuned_checkpoint": WEIGHTS_ROOT / "finetuned_weights/expert3_best.pt"},
    "expert4": {"beats_checkpoint": WEIGHTS_ROOT / "pretrained/BEATs_iter3_plus_AS2M.pt", "clap_checkpoint": WEIGHTS_ROOT / "pretrained/laion-clap.pt", "finetuned_checkpoint": WEIGHTS_ROOT / "finetuned_weights/expert4_best.pt"},
}
EXPECTED_COUNTS = {"train": 7500, "validation": 3000, "test": 3000}


## 3. Môi trường Kaggle và GPU
Bật GPU trước khi chạy. Không tự cài lại PyTorch/CUDA. Chẩn đoán này dừng rõ ràng nếu không có GPU.


In [ ]:
import sys, os, json, subprocess, shutil
import torch
print("Python:", sys.version)
print("PyTorch:", torch.__version__, "CUDA runtime:", torch.version.cuda)
if not torch.cuda.is_available():
    raise RuntimeError("Chưa có CUDA. Hãy chọn GPU trong phần cài đặt Accelerator của Kaggle.")
print("GPU:", torch.cuda.get_device_name(0))
free, total = torch.cuda.mem_get_info(0)
print(f"VRAM trống/tổng: {free / 2**30:.2f}/{total / 2**30:.2f} GiB")
print("Kaggle Inputs:", [str(p) for p in sorted(INPUT_ROOT.iterdir())])
print("Dung lượng đĩa trống GiB:", shutil.disk_usage("/kaggle/working").free / 2**30)


## 4. Mã nguồn và phụ thuộc
Ưu tiên source đính kèm hoặc thư mục đã có. Chỉ clone khi thiếu source và người dùng cho phép. Nếu clone chưa chứa script mới, hãy đính kèm đúng phiên bản repository đã triển khai notebook này.

`INSTALL_REQUIREMENTS=True` cài `requirements-kaggle.txt` trước khi import pipeline. Installer khóa các phiên bản Torch/CUDA đang có; chế độ offline dùng `WHEELHOUSE`. Nếu thư viện đã nạp vừa đổi phiên bản, khởi động lại kernel rồi chạy từ đầu theo thông báo.


In [ ]:
if not PROJECT_ROOT.exists():
    if REPOSITORY_SOURCE:
        shutil.copytree(REPOSITORY_SOURCE, PROJECT_ROOT, ignore=shutil.ignore_patterns("__pycache__", ".cache", ".venv"))
    elif ALLOW_REPOSITORY_DOWNLOAD:
        subprocess.run(["git", "clone", REPO_URL, str(PROJECT_ROOT)], check=True)
        if REPO_REVISION:
            subprocess.run(["git", "-C", str(PROJECT_ROOT), "checkout", "--detach", REPO_REVISION], check=True)
    else:
        raise FileNotFoundError("Đặt REPOSITORY_SOURCE tới source MOESCORE tại Kaggle Input.")
required_scripts = ["scripts/extract_moe_feature_store.py", "scripts/kaggle_moe_pipeline.py",
                    "scripts/train_moe_gating.py", "training/moe_evaluator.py",
                    "scripts/install_kaggle_dependencies.py", "scripts/prepare_moe_text_cache.py", "requirements-kaggle.txt"]
for name in required_scripts:
    if not (PROJECT_ROOT / name).is_file():
        raise FileNotFoundError(f"Source chưa đủ thành phần: {PROJECT_ROOT / name}")
sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)
git = subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True)
print("Source Git SHA:", git.stdout.strip() if git.returncode == 0 else "Bản source không có .git")
if INSTALL_REQUIREMENTS or EXTRA_PACKAGES:
    import importlib, importlib.metadata
    loaded_before = {}
    for module, distribution in (("numpy", "numpy"), ("scipy", "scipy"), ("pandas", "pandas"),
                                 ("transformers", "transformers"), ("timm", "timm"), ("laion_clap", "laion-clap")):
        if module in sys.modules:
            try:
                loaded_before[distribution] = importlib.metadata.version(distribution)
            except importlib.metadata.PackageNotFoundError:
                pass
    command = [sys.executable, str(PROJECT_ROOT / "scripts/install_kaggle_dependencies.py"),
               "--requirements", str(PROJECT_ROOT / "requirements-kaggle.txt"),
               "--constraints", "/kaggle/working/moe_runtime_constraints.txt",
               "--report-json", "/kaggle/working/moe_dependency_install.json"]
    if not INSTALL_REQUIREMENTS:
        command += ["--no-requirements"]
    if WHEELHOUSE:
        command += ["--wheelhouse", str(WHEELHOUSE)]
    for package in EXTRA_PACKAGES:
        command += ["--extra", package]
    subprocess.run(command, check=True)
    importlib.invalidate_caches()
    changed_loaded = [name for name, version in loaded_before.items()
                      if importlib.metadata.version(name) != version]
    if changed_loaded:
        raise RuntimeError("Đã cài thư viện, nhưng các gói đang được Python sử dụng đã đổi phiên bản: "
                           + ", ".join(changed_loaded)
                           + ". Khởi động lại kernel Python rồi chạy lại từ đầu; không tạo phiên Kaggle mới.")
from scripts import kaggle_moe_pipeline as pipeline
def run_script(*args):
    print("Chạy:", sys.executable, *map(str, args), flush=True)
    subprocess.run([sys.executable, *map(str, args)], cwd=PROJECT_ROOT, check=True)


## 5. Tìm bộ dữ liệu XACLE
Chỉ nhận root có cả `meta_data/` và `wav/`. Nếu tìm thấy nhiều root, điền `DATASET_ROOT` tường minh.


In [ ]:
DATASET_ROOT = pipeline.discover_dataset(INPUT_ROOT, DATASET_ROOT)
print("Dataset:", DATASET_ROOT)
if HF_CACHE_DIR is None:
    candidates = [p for pattern in ("*", "*/hub", "*/huggingface/hub")
                  for p in INPUT_ROOT.glob(pattern)
                  if (p / "models--roberta-base").is_dir()
                  and (p / "models--google-bert--bert-base-uncased").is_dir()]
    candidates = sorted(set(candidates))
    if len(candidates) > 1:
        raise ValueError(f"Có nhiều cache. Điền HF_CACHE_DIR: {candidates}")
    HF_CACHE_DIR = candidates[0] if candidates else Path("/kaggle/working/hf_cache")
os.environ.update(HF_HOME=str(HF_CACHE_DIR), HF_HUB_CACHE=str(HF_CACHE_DIR),
                  TRANSFORMERS_CACHE=str(HF_CACHE_DIR))
os.environ.update(HF_HUB_OFFLINE="0" if ALLOW_CACHE_DOWNLOAD else "1",
                  TRANSFORMERS_OFFLINE="0" if ALLOW_CACHE_DOWNLOAD else "1",
                  HF_DATASETS_OFFLINE="1")


## 6. Schema, số lượng và đường dẫn dữ liệu
Kiểm tra đủ 7.500/3.000/3.000 cặp, file WAV, caption, ID duy nhất và nhãn train/validation. Các giá trị nhãn test được loại khỏi metadata đầu vào; chỉ đọc trong bước đánh giá test cuối cùng.


## 7. Checkpoint của bốn chuyên gia
Xác nhận đủ tám file ở đúng tên đã chốt. Kiểm tra SHA, schema và restoration nghiêm ngặt tiếp tục được thực hiện bởi extractor/factory.


In [ ]:
for expert, fields in EXPERT_CHECKPOINTS.items():
    for field, path in fields.items():
        if not path.is_file() or path.stat().st_size == 0:
            raise FileNotFoundError(f"{expert}.{field}: thiếu hoặc rỗng: {path}")
        print(expert, field, path, f"{path.stat().st_size / 2**30:.3f} GiB")


## 8. Nguồn expert, UniLM và cấu hình
E2/E3 dùng source trong repository. E4 dùng UniLM đi kèm: chỉ chấp nhận khác biệt hai import tương đối khi khôi phục đúng fingerprint nguồn huấn luyện gốc. Không thay kiến trúc BEATs hoặc bỏ kiểm tra checkpoint.

Cache cần chứa đúng `roberta-base`, `bert-base-uncased` (config/tokenizer cho E2) và `google-bert/bert-base-uncased` (cho E3). Tên BERT ngắn và tên có namespace phải được kiểm riêng. Chỉ tải khi bật `ALLOW_CACHE_DOWNLOAD`; extraction luôn offline sau chuẩn bị.


In [ ]:
EXPERT_CONFIG, TRAIN_CONFIG, resolved_experts = pipeline.generate_configs(
    PROJECT_ROOT, DATASET_ROOT, WEIGHTS_ROOT, HF_CACHE_DIR, FEATURE_STORE_ROOT,
    OUTPUT_DIR, Path("/kaggle/working"), expert_checkpoints=EXPERT_CHECKPOINTS)
preflight_report = pipeline.preflight(resolved_experts, DATASET_ROOT,
    Path("/kaggle/working/moe_preflight"), allow_download=ALLOW_CACHE_DOWNLOAD)
print("Phụ thuộc:", pipeline.check_dependencies())
print(json.dumps(preflight_report, indent=2, ensure_ascii=False))
from training.config import load_moe_training_config
training_config = load_moe_training_config(TRAIN_CONFIG)
print(Path(TRAIN_CONFIG).read_text())


## 9. Khởi tạo và phục hồi expert
Khi đến extraction, mỗi subprocess gọi `create_expert_adapter()` một lần, phục hồi pretrained/base rồi fine-tuned, xác nhận frozen/eval và module origin. Tiến trình kết thúc trước khi nạp expert tiếp theo. Báo cáo thực tế nằm ở `_extraction/expert*_runtime.json`. Chế độ `preflight` không dựng expert.


## 10. Cấu hình extraction và khôi phục artifact
Batch size riêng từng expert, FP32, shard nguyên tử và kiểm tra checksum. Giữ `SHARD_SIZE` khi resume; có thể giảm batch inference sau OOM. Store/input từ phiên trước phải có manifest và checksum đầy đủ.


In [ ]:
from data.moe_feature_store import validate_feature_store
if EXISTING_FEATURE_STORE_ROOT:
    source = Path(EXISTING_FEATURE_STORE_ROOT)
    for split in EXPECTED_COUNTS:
        pipeline.verified_source_manifest(source, split)
    if FEATURE_STORE_ROOT.exists() and any(FEATURE_STORE_ROOT.iterdir()):
        for split in EXPECTED_COUNTS:
            if validate_feature_store(source, split=split)["store_identity_sha256"] != validate_feature_store(FEATURE_STORE_ROOT, split=split)["store_identity_sha256"]:
                raise ValueError("Store đích đã có nội dung khác; dùng thư mục mới.")
    else:
        shutil.copytree(source, FEATURE_STORE_ROOT, dirs_exist_ok=True)
if RESTORE_OUTPUT_ROOT:
    if OUTPUT_DIR.exists() and any(OUTPUT_DIR.iterdir()):
        raise ValueError("OUTPUT_DIR đã có nội dung; chọn output trống trước khi khôi phục.")
    shutil.copytree(RESTORE_OUTPUT_ROOT, OUTPUT_DIR, dirs_exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


## 11. Đặc trưng train
Train dùng preprocessing đánh giá xác định: E1 center crop khi dài, các nhánh khác giữ đúng chính sách gốc. Đây là lựa chọn cache frozen, không tương đương random crop online mỗi epoch. Chỉ store train mang nhãn vào optimizer.


## 12. Đặc trưng validation
Tách store validation, giữ nguyên thứ tự và nhãn trung bình gốc. Chỉ dùng trong đánh giá, scheduler và chọn checkpoint theo SRCC.


## 13. Đặc trưng test và chạy extraction
Test store chỉ chứa ID, điểm và feature; không có nhãn. Lệnh dưới thực hiện E1 → cả ba split, rồi E2, E3, E4. Không chạy ba lần nạp lại từng expert. `smoke` dùng root riêng, tối đa bốn mẫu mỗi split.


In [ ]:
if RUN_MODE in {"smoke", "extract", "full"}:
    extraction_root = Path(str(FEATURE_STORE_ROOT) + "_smoke") if RUN_MODE == "smoke" else FEATURE_STORE_ROOT
    options = ["--smoke"] if RUN_MODE == "smoke" else []
    run_script("scripts/extract_moe_feature_store.py",
        "--config", EXPERT_CONFIG, "--dataset-root", DATASET_ROOT,
        "--output-root", extraction_root, "--device", "cuda:0",
        "--batch-sizes", ",".join(str(EXTRACTION_BATCH_SIZES[e]) for e in ("expert1","expert2","expert3","expert4")),
        "--shard-size", str(SHARD_SIZE), *options)
    if RUN_MODE == "smoke":
        smoke_report = pipeline.smoke_store(extraction_root, training_config)
        (OUTPUT_DIR / "smoke_report.json").write_text(json.dumps(smoke_report, indent=2, allow_nan=False))
        print(json.dumps(smoke_report, indent=2))
        print("Đã chạy thử giới hạn; chưa hoàn tất extraction/huấn luyện toàn tập.")


## 14. Xác minh feature store
Mỗi split phải đủ mẫu, FP32 `[N,4]` và `[N,7936]`, đúng E1/E2/E3/E4, checksum và provenance. Không dùng store smoke để huấn luyện đầy đủ.


In [ ]:
if RUN_MODE in {"extract", "train", "resume", "evaluate", "full"}:
    from scripts.extract_moe_feature_store import prepare_job, completed_stores_match
    if RUN_MODE in {"train", "resume", "evaluate"}:
        _, job, manifests = prepare_job(EXPERT_CONFIG, DATASET_ROOT, FEATURE_STORE_ROOT,
            batch_sizes=EXTRACTION_BATCH_SIZES, shard_size=SHARD_SIZE)
        if not completed_stores_match(job, manifests, FEATURE_STORE_ROOT):
            raise FileNotFoundError("Feature store chưa đầy đủ. Chạy extract hoặc full trước.")
    for split, expected in EXPECTED_COUNTS.items():
        manifest = validate_feature_store(FEATURE_STORE_ROOT, split=split, require_labels=split != "test")
        count = sum(s["count"] for s in manifest["shards"])
        if count != expected or (split == "test" and manifest["labels_present"]):
            raise ValueError(f"{split}: sai số lượng hoặc test store chứa nhãn.")
        pipeline.verified_source_manifest(FEATURE_STORE_ROOT, split)
        print(split, count, manifest["store_identity_sha256"])


## 15. Huấn luyện Gating Network
Gọi trainer hiện có: AdamW, scheduler, loss ClippedMSE + 0.5 Contrastive + 0.2 Diversity. Gate nhận 7936 feature và tạo bốn weight phụ thuộc mẫu. Target được đổi `(raw-5)/5` tại loss boundary. Chỉ gate cập nhật tham số.


## 16. Quản lý checkpoint và resume
`latest_gate.pt` chứa trạng thái optimizer/scheduler/RNG/sampler để tiếp tục. `best_gate.pt` được chọn duy nhất bằng SRCC validation. `full` tự tiếp tục từ latest nếu đã có; `train` từ chối ghi đè một lượt chạy có checkpoint.


In [ ]:
if RUN_MODE in {"train", "resume", "full"}:
    latest = OUTPUT_DIR / "latest_gate.pt"
    resume = Path(RESUME_CHECKPOINT) if RESUME_CHECKPOINT else latest
    options = []
    if RUN_MODE == "resume" or (RUN_MODE == "full" and latest.exists()):
        if not resume.is_file():
            raise FileNotFoundError(f"Thiếu checkpoint resume: {resume}")
        options = ["--resume", str(resume)]
    elif latest.exists() or (OUTPUT_DIR / "best_gate.pt").exists():
        raise FileExistsError("Output đã có checkpoint. Chọn mode resume hoặc output mới.")
    shutil.copy2(TRAIN_CONFIG, OUTPUT_DIR / "training_config.yaml")
    run_script("scripts/train_moe_gating.py", "--config", TRAIN_CONFIG,
               "--report-json", OUTPUT_DIR / "training_run.json", *options)
    trained = pipeline.record_training(OUTPUT_DIR / "training_run.json", OUTPUT_DIR)
    print("Best checkpoint:", trained["best_checkpoint"], "Best validation SRCC:", trained["best_srcc_raw_scale"])


## 17. Đánh giá validation bằng best gate
Kiểm tra provenance và chính xác identity store validation đã dùng chọn checkpoint. Đánh giá toàn bộ split ở chế độ eval/no-grad; không làm tròn prediction trước metric.


## 18. Suy luận và đánh giá test cuối cùng
Bước này chạy sau huấn luyện. Lưu prediction test đầy đủ độ chính xác trước, sau đó mới đọc `average_score` từ test CSV và ghép theo sample ID, filename, caption. Gọi lại `compute_metrics()` hiện có. Submission hai chữ số được lưu riêng.


In [ ]:
final_metrics = None
if RUN_MODE in {"train", "resume", "evaluate", "full"}:
    best = OUTPUT_DIR / "best_gate.pt"
    if not best.is_file():
        raise FileNotFoundError("Chưa có best_gate.pt; kiểm tra SRCC validation có được định nghĩa hay không.")
    final_metrics = pipeline.evaluate_best(training_config, DATASET_ROOT, checkpoint=best)
    import pandas as pd
    table = pd.DataFrame([{"Split": split, **{name: final_metrics[split][name] for name in ("SRCC","LCC","KTAU","MSE")}}
                          for split in ("validation","test")])
    display(table.rename(columns={"SRCC":"SRCC ↑", "LCC":"LCC ↑", "KTAU":"KTAU ↑", "MSE":"MSE ↓"}))


## 19. Biểu đồ huấn luyện và đánh giá
Chỉ vẽ dữ liệu log thật. Giá trị tương quan không xác định giữ dạng rỗng; không điền số giả.


In [ ]:
if RUN_MODE in {"train", "resume", "evaluate", "full"}:
    import matplotlib.pyplot as plt
    history_path = OUTPUT_DIR / "training_history.json"
    if history_path.exists():
        history = json.loads(history_path.read_text())
        if history:
            fig, axes = plt.subplots(1, 2, figsize=(12, 4))
            axes[0].plot([h["epoch"] + 1 for h in history], [h["train_loss"] for h in history], label="Train loss")
            axes[0].plot([h["epoch"] + 1 for h in history], [h["validation_loss_normalized"] for h in history], label="Validation loss")
            axes[0].legend()
            axes[1].plot([h["epoch"] + 1 for h in history], [h["validation_metrics_raw"]["srcc"] for h in history])
            axes[1].set_ylabel("Validation SRCC")
            for ax in axes: ax.set_xlabel("Epoch")
            fig.tight_layout()
            fig.savefig(OUTPUT_DIR / "training_curves.png", dpi=150)
            plt.show()
    if final_metrics:
        for split in ("validation", "test"):
            print(split, {k: final_metrics[split][k] for k in ("SRCC", "LCC", "KTAU", "MSE")})


## 20. Xuất và bảo toàn kết quả
`/kaggle/input` chỉ đọc. `/kaggle/working` cần được lưu bằng chức năng lưu output phiên Kaggle hoặc tải về; không giả định thư mục này tồn tại ở phiên mới. Có thể đóng gói store riêng để đính kèm lại. Chỉ archive sau khi xác minh store hoàn chỉnh.


In [ ]:
if RUN_MODE in {"train", "resume", "evaluate", "full"}:
    required = ["best_gate.pt", "metrics.json", "validation_metrics.json", "test_metrics.json",
                "validation_predictions.csv", "test_predictions.csv"]
    for name in required:
        if not (OUTPUT_DIR / name).is_file():
            raise FileNotFoundError(f"Thiếu artifact cuối: {name}")
    print("Artifact:", [(p.name, p.stat().st_size) for p in sorted(OUTPUT_DIR.iterdir()) if p.is_file()])
    archive = shutil.make_archive("/kaggle/working/moe_gating_results", "zip", OUTPUT_DIR)
    print("Tệp kết quả:", archive)
if SAVE_FEATURE_STORE_ARCHIVE and RUN_MODE in {"extract", "train", "resume", "evaluate", "full"}:
    for split in EXPECTED_COUNTS:
        validate_feature_store(FEATURE_STORE_ROOT, split=split)
    print("Đóng gói store; cần đủ dung lượng trống cho bản sao archive.")
    print(shutil.make_archive("/kaggle/working/moe_feature_store", "zip", FEATURE_STORE_ROOT))
print("Hoàn tất chế độ:", RUN_MODE)
